## Pre-Built Invoice Processing with Azure AI Content Understanding

### Installing Required Libraries

In [ ]:
%pip install python-dotenv azure-ai-documentintelligence

### Setting up the Environment

Create a `.env` file with the endpoint and key for an Azure AI Content Understanding resource. The endpoint should look like `https://<resource-name>.services.ai.azure.com/`.

```text
FOUNDRY_AI_SERVICES_ENDPOINT=https://<resource-name>.services.ai.azure.com/
FOUNDRY_AI_SERVICES_KEY=<your-resource-key>
```

Before using `prebuilt-invoice`, deploy `gpt-4.1` and `text-embedding-3-large` in the Foundry resource and configure the default model deployments once. See Microsoft's [model deployment setup sample](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-contentunderstanding_1.1.0/sdk/contentunderstanding/azure-ai-contentunderstanding/samples/sample_update_defaults.py).

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest

load_dotenv()

endpoint = os.getenv("FOUNDRY_AI_SERVICES_ENDPOINT")
api_version = "2024-11-30"

### Creating the Content Understanding Client

In [2]:
from azure.identity import DefaultAzureCredential

# Authenticate with Entra ID (run `az login` first)
credential = DefaultAzureCredential()


client = DocumentIntelligenceClient(
    endpoint=endpoint,
    credential= credential,
    api_version=api_version,
)

### Analyzing the Local Sample Invoice

In [3]:
sample_path = Path("Samples") / "sample-invoice.pdf"
if not sample_path.is_file():
    sample_path = (
        Path("Information-Extraction")
        / "Document-Intelligence"
        / "Pre-Built-Offerings"
        / "Samples"
        / "sample-invoice.pdf"
    )

if not sample_path.is_file():
    raise FileNotFoundError(
        "Could not find Samples/sample-invoice.pdf. Run this notebook from its folder or the repository root."
    )

sample_bytes = sample_path.read_bytes()

In [4]:
poller = client.begin_analyze_document(
    model_id="prebuilt-invoice",
    body=AnalyzeDocumentRequest(
        bytes_source=sample_bytes,
    )
)
result = poller.result()

if not result.documents:
    raise ValueError("The invoice analysis returned no document content.")


In [11]:

for idx, invoice in enumerate(result.documents):
    fields = invoice.fields or {}

    customer_name = fields.get("CustomerName")
    invoice_date = fields.get("InvoiceDate")
    invoice_total = fields.get("SubTotal")
    amount_due = fields.get("AmountDue")
    previous_balance = fields.get("PreviousUnpaidBalance")
    print("Customer name:", customer_name.value_string if customer_name else "N/A")
    print("Invoice date:", invoice_date.value_date if invoice_date else "N/A")
    print("Invoice total:", invoice_total.content if invoice_total else "N/A")
    print("Amount due:", amount_due.content if amount_due else "N/A")
    print("Previous balance:", previous_balance.content if previous_balance else "N/A")
    print("-" * 40)


Customer name: MICROSOFT CORPORATION
Invoice date: 2019-11-15
Invoice total: $100.00
Amount due: $610.00
Previous balance: $500.00
----------------------------------------


In [12]:
for name, field in invoice.fields.items():
    field_value = field.value_string if field.value_string else field.content
    print("type: '{}' name: '{}' value: '{}' confidence: {}".format(
        field.type, name, field_value, field.confidence))
print("-" * 40)

type: 'DocumentFieldType.CURRENCY' name: 'AmountDue' value: '$610.00' confidence: 0.938
type: 'DocumentFieldType.ADDRESS' name: 'BillingAddress' value: '123 Bill St,
Redmond WA, 98052' confidence: 0.889
type: 'DocumentFieldType.STRING' name: 'BillingAddressRecipient' value: 'Microsoft Finance' confidence: 0.938
type: 'DocumentFieldType.ADDRESS' name: 'CustomerAddress' value: '123 Other St,
Redmond WA, 98052' confidence: 0.89
type: 'DocumentFieldType.STRING' name: 'CustomerAddressRecipient' value: 'Microsoft Corp' confidence: 0.938
type: 'DocumentFieldType.STRING' name: 'CustomerId' value: 'CID-12345' confidence: 0.968
type: 'DocumentFieldType.STRING' name: 'CustomerName' value: 'MICROSOFT CORPORATION' confidence: 0.892
type: 'DocumentFieldType.DATE' name: 'DueDate' value: '12/15/2019' confidence: 0.971
type: 'DocumentFieldType.DATE' name: 'InvoiceDate' value: '11/15/2019' confidence: 0.971
type: 'DocumentFieldType.STRING' name: 'InvoiceId' value: 'INV-100' confidence: 0.971
type: 'Docu

In [14]:
import pprint
pprint.pp(fields)

{'AmountDue': {'type': 'currency', 'valueCurrency': {'currencySymbol': '$', 'amount': 610.0, 'currencyCode': 'USD'}, 'content': '$610.00', 'boundingRegions': [{'pageNumber': 1, 'polygon': [7.3664, 7.8028, 7.9341, 7.8045, 7.9338, 7.9676, 7.3669, 7.9706]}], 'confidence': 0.938, 'spans': [{'offset': 653, 'length': 7}]},
 'BillingAddress': {'type': 'address', 'content': '123 Bill St,\nRedmond WA, 98052', 'boundingRegions': [{'pageNumber': 1, 'polygon': [0.5765, 4.358, 2.0309, 4.3578, 2.0309, 4.7167, 0.5765, 4.7168]}], 'confidence': 0.889, 'spans': [{'offset': 328, 'length': 12}, {'offset': 370, 'length': 17}], 'valueAddress': {'houseNumber': '123', 'road': 'Bill St', 'postalCode': '98052', 'city': 'Redmond', 'state': 'WA', 'streetAddress': '123 Bill St'}},
 'BillingAddressRecipient': {'type': 'string', 'valueString': 'Microsoft Finance', 'content': 'Microsoft Finance', 'boundingRegions': [{'pageNumber': 1, 'polygon': [0.5763, 4.1583, 1.8097, 4.1593, 1.8096, 4.3114, 0.5762, 4.3104]}], 'conf

In [15]:
pprint.pp(result)

{'apiVersion': '2024-11-30', 'modelId': 'prebuilt-invoice', 'stringIndexType': 'textElements', 'content': 'CONTOSO LTD.\nINVOICE\nContoso Headquarters\nINVOICE: INV-100\n123 456th St\nDATE: 11/15/2019\nNew York, NY, 10001\nDUE DATE: 12/15/2019\nCUSTOMER NAME: MICROSOFT CORPORATION\nCUSTOMER ID: CID-12345\nMicrosoft Corp\n123 Other St,\nRedmond WA, 98052\nBILL TO:\nSHIP TO:\nSERVICE ADDRESS:\nMicrosoft Finance\nMicrosoft Delivery\nMicrosoft Services\n123 Bill St,\n123 Ship St,\n123 Service St,\nRedmond WA, 98052\nRedmond WA, 98052\nRedmond WA, 98052\nSALESPERSON\nP.O. NUMBER\nREQUISITIONER\nSHIPPED VIA\nF.O.B. POINT\nTERMS\nPO-3333\nQUANTITY\nDESCRIPTION\nUNIT PRICE\nTOTAL\n1\nTest for 23 fields\n1\n$100.00\nSUBTOTAL\n$100.00\nSALES TAX\n$10.00\nTOTAL\n$110.00\nPREVIOUS BALANCE\n$500.00\nTOTAL DUE\n$610.00\nTHANK YOU FOR YOUR BUSINESS!\nREMIT TO:\nContoso Billing\n123 Remit St\nNew York, NY, 10001', 'pages': [{'pageNumber': 1, 'angle': -0.017000000923871994, 'width': 8.5, 'height': 11, 